In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 7 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# Week 10 produced a large negative calibration miss (~ -3.7σ).
#
# Therefore:
# - refit including Week 10
# - centre search on the ACTUAL incumbent
# - use a conservative empirical trust region
# - compare mean, EI and UCB
# - do NOT automatically expand on boundary hits
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 10 calibration check
# ------------------------------------------------------------

week10_pred_mean = 3.1063997281066196
week10_pred_std = 0.07173493359439305
week10_actual = 2.8408984660368275

week10_error = (
    week10_actual
    - week10_pred_mean
)

week10_z_error = (
    week10_error
    / week10_pred_std
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week10_pred_mean)
print("Predicted std :", week10_pred_std)
print("Actual        :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(6) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Severe Week 10 miss -> conservative cap

empirical_cap = min(
    1.20 * nearest_distance,
    0.06
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest observed point:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 6. Conservative ARD trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.16 * lengthscales,
    0.015,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 11 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower:")
print(lower)

print("\nUpper:")
print(upper)


# ------------------------------------------------------------
# 7. Dense trust-region candidate search
# ------------------------------------------------------------

rng = np.random.default_rng(42)

candidates = rng.uniform(
    lower,
    upper,
    size=(450000, 6)
)

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 8. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 10. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 13. Distance from actual incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 14. Trust-region boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(len(x)):

        if abs(
            x[j] - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j] - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )

DATA
X shape: (40, 6)
Y shape: (40,)

Current best:
[0.16523  0.227995 0.587283 0.251481 0.277646 0.674437] -> 3.0575674596116973

Y range:
min = 0.0027014650245082332
max = 3.0575674596116973
std = 1.0001855237535264

WEEK 10 CALIBRATION CHECK
Predicted mean: 3.1063997281066196
Predicted std : 0.07173493359439305
Actual        : 2.8408984660368275

Prediction error:
-0.2655012620697921

Error / predicted std:
-3.701143205498753


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.634**2 * Matern(length_scale=[0.481, 0.356, 0.857, 0.401, 0.257, 0.381], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[0.48132951 0.35626618 0.85690156 0.40051839 0.25722325 0.38110401]

Normalised inverse-lengthscale sensitivity:
[0.13795475 0.18638225 0.07749045 0.16578937 0.25814809 0.17423509]

EMPIRICAL LOCAL SCALE
Nearest observed point:
0.12029264144992408

Empirical cap:
0.06

WEEK 11 TRUST REGION
Centre:
[0.16523  0.227995 0.587283 0.251481 0.277646 0.674437]

Half-widths:
[0.06       0.05700259 0.06       0.06       0.04115572 0.06      ]

Lower:
[0.10523    0.17099241 0.527283   0.191481   0.23649028 0.614437  ]

Upper:
[0.22523    0.28499759 0.647283   0.311481   0.31880172 0.734437  ]

Candidates after duplicate filtering:
450000

PRIMARY EI
candidate = [0.17513585 0.26936859 0.59794798 0.25579713 0.29342097 0.66660269]
mean = 3.090627488991685
std = 0.07677380288667247
EI = 0.04995495719006144

EI SENSITIVITY

xi = 0.000000e+00 
 c

In [2]:
# ============================================================
# FINAL FUNCTION 7 - WEEK 11 SELECTION
# ============================================================

final_idx = np.argmax(mu)

week11_candidate = candidates[final_idx]

print("Week 11 Function 7 candidate:")
print(week11_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week11_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week11_candidate
)

print("\nPortal format:")
print(portal)

Week 11 Function 7 candidate:
[0.17493106 0.25767125 0.57701987 0.26231824 0.28719574 0.66910269]

Predicted mean:
3.095186224163146

Predicted std:
0.04889060270245136

Distance from current best:
0.03629353031658894

Portal format:
0.174931-0.257671-0.577020-0.262318-0.287196-0.669103
